# Case 1: Unsupervised Learning in Equity Returns

## Set-up

Imagine you work on an analytics team at an asset-management firm.

The firm trades a diversified portfolio of equities. Beyond forecasting a single stock's return, you are often asked questions like:

- **Risk decomposition:** "What are the main sources of co-movement across our holdings?"
- **Diversification:** "Are we truly diversified, or are many names effectively moving together?"
- **Monitoring:** "If the market regime changes, how would we detect it quickly?"

Unsupervised learning methods are natural tools for these tasks because they help you compress a high-dimensional panel of returns into a small number of summarizing components (e.g., PCA factors) and to discover structure in the cross-section (e.g., clusters of stocks that behave similarly).

In this case you will build a baseline PCA factor model, then use diagnostics and additional unsupervised methods to look for potentially meaningful structure in the data.

### Things to keep in mind

Interpretability is not automatic. PCA components are statistical objects; economic meaning is something you must argue for.

Nonstationarity matters. The latent structure can shift over time (crises, policy changes, sector rotations).

Data leakage matters if you later use these objects for prediction. If you build factors on the full sample and then feed them into a supervised model, you may inadvertently use information from the future.



## Constructing the Return Matrix

In the first problem set, we constructed a return matrix for a cross-section of equities. We follow the same general approach here.



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, FactorAnalysis
import seaborn as sns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 500)

In [ ]:
#Config

startdate = '2011-01-01' # Change this to your desired start date (Minimum: 2011-01-01)
enddate = '2024-12-31'  # Change this to your desired end date (Maximum: 2024-12-31)

In [ ]:
!wget -O df_factor.csv https://www.dropbox.com/scl/fi/glm90k03u7yehadyw9dk5/df_factor.csv?rlkey=iwhnsi7sf9ogsvfpm9jltqzew&dl=0

df = pd.read_csv('df_factor.csv', parse_dates=['date'])

tic_list= df.loc[df['date'] < startdate, 'tic'].unique()

df = df[df['tic'].isin(tic_list[:100])]


In [ ]:
df = df[['date', 'tic', 'ret']]

In [ ]:
df.sort_values('date', inplace=True)

In [ ]:
returns_matrix = df.pivot(index='date', columns='tic', values='ret')
returns_matrix = returns_matrix.dropna()

## Standardization

Before PCA, we typically standardize each column (ticker) so that PCA focuses on *correlation-style* co-movement rather than simply variance differences driven by a few volatile assets.


In [ ]:
returns_standardized = (returns_matrix - returns_matrix.mean()) / returns_matrix.std()  #Z score Standardization


---

**Question 1.** Why might standardization make sense when looking at unsupervised learning in a cross-section of equities? E.g., what would happen if one ticker had much higher volatility than the others.

When might you *not* want to standardize before PCA (or applying other unsupervised learning methods)?

If you wanted to use unsupervised learning as part of a supervised learning pipeline that includes train/test splits, what mistake has been made in the above code that does the standardization?

---

## Baseline PCA

We now look at the first two principal components extracted from PCA applied to the standardized return matrix.

At this stage, the choice of two components is arbitrary. It is not meant to be optimal or economically justified. Instead, this serves as a baseline specification that we will later revisit and improve.

PCA extracts directions in the data that explain the largest share of cross-sectional variance. The resulting components are purely statistical objects. Whether they correspond to economically meaningful forces is something you will need to evaluate.

From the PCA output, we construct two objects:

- The **factor loadings**, which describe how each stock loads on each extracted component.
- The **factor time series**, which describe how the extracted components evolve over time.

In [ ]:
pca = PCA(n_components=2) # n_componments = number of factors
factors = pca.fit_transform(returns_standardized)

In [ ]:
factor_loadings = pd.DataFrame(pca.components_.T,
                               index=returns_standardized.columns,
                               columns=["Factor1", "Factor2"])

factors_df = pd.DataFrame(factors, index=returns_standardized.index,
                          columns=["Factor1", "Factor2"])

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(factor_loadings, annot=False, cmap="coolwarm", center=0)
plt.title("Factor Loadings (Stock Sensitivities)")
plt.show()

---

**Question 2.** The code block above produces a heatmap showing the factor loadings. Examine the heatmap of factor loadings carefully. Do the exposures suggest an economic story (e.g., a market-wide factor, growth vs value, tech vs non. tech)? What additional data would you want to validate your story (e.g., sector labels, fundamentals, ...)?

---

In [ ]:
factors_df.plot(figsize=(12,6), title="Extracted Latent Factors")
plt.show()

---

**Question 3.** Now we present the time-series plots of the two extracted components. Consider their volatility, persistence, and behavior during turbulent periods. Do the factors move together, or do they appear to capture distinct sources of variation?

---

## Exploring Factor Behavior at a More Granular Level

The next two cells allow you to inspect the extracted factors more closely.

First, you can isolate a specific year and examine how the latent factors behave within that shorter window. This can be useful for identifying regime changes, crisis dynamics, or shifts in factor dominance that may not be obvious in the full-sample plot.

Second, you can overlay the return of a specific asset on top of one or more extracted factors. This provides a more detailed view of how individual securities co-move with the latent components. While this is not a formal regression, it can help build intuition about explanatory power and exposure patterns.

In [ ]:
year = 2020 # Change this to the year you want to visualize

factors_df[factors_df.index.year == year].plot(
    figsize=(12,6),
    title=f"Extracted Latent Factors — {year}"
)

plt.show()

In [ ]:
year = 2020 # Change this to the year you want to visualize
asset = "AAPL" # Change this to the asset you want to visualize
factor_choice = "All"   # "Factor1" / "Factor2" / "All"

ymin = factors_df.loc[str(year)].min().min()
ymax = factors_df.loc[str(year)].max().max()
padding = 0.05 * (ymax - ymin)

cols = factors_df.columns if factor_choice == "All" else [factor_choice]

ax = factors_df.loc[str(year), cols].plot(
    figsize=(12,6),
    title=f"{factor_choice} + {asset} Returns — {year}"
)

ax.set_ylim(ymin - padding, ymax + padding)

returns_matrix.loc[str(year), asset].plot(
    ax=ax,
    secondary_y=True,
    label=asset,
    color="black",
    alpha=0.5,
    mark_right=False
)

ax.legend(ax.get_legend_handles_labels()[0] + ax.right_ax.get_legend_handles_labels()[0],
          ax.get_legend_handles_labels()[1] + ax.right_ax.get_legend_handles_labels()[1])

plt.show()


---

**Question 4.** Use these visualizations thoughtfully. Vary the year, the asset, and the selected factor(s).

- Does the relationship between an asset and a factor appear stable across time?
- Does one factor seem to matter more for certain assets?
- Are there periods where the factors fail to track asset behavior?

---


## Clustering: do groups of stocks behave similarly?

PCA compresses the panel into a few dimensions. We can then cluster stocks based on their factor loadings.

One simple approach:

- represent each ticker by its loadings on the first `N_FACTORS` components,
- apply a clustering algorithm (e.g., k-means),
- see whether clusters look “sensible” (potentially corresponding to industries, styles, or risk exposures).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Build the feature matrix for clustering: each ticker is represented by its loadings
X_cluster = factor_loadings.values

# Arbitrarily choose K = 4
K = 4

km = KMeans(n_clusters=K, random_state=0, n_init=50)
cluster_labels = km.fit_predict(X_cluster)

clusters = pd.Series(cluster_labels, index=factor_loadings.index, name='cluster')
print(clusters.value_counts().sort_index())

# Show a few tickers per cluster
for k in range(K):
    members = clusters[clusters == k].index.tolist()
    print(f"\nCluster {k}: n = {len(members)}")
    print(members[:20])


In [ ]:
# 2D scatter of loadings on the two factors
plt.figure(figsize=(8,6))
plt.scatter(factor_loadings.iloc[:,0], factor_loadings.iloc[:,1], c=cluster_labels)
plt.title('Clusters visualized in (loading on Factor 1, loading on Factor 2) space')
plt.xlabel('Loading on Factor 1')
plt.ylabel('Loading on Factor 2')
plt.show()



---

**Question 5.** Pick **one** cluster and investigate it: do the tickers look like they come from similar industries? (You can use the web, Bloomberg/Yahoo, or AI tools to quickly look up what the tickers are.)

 How would you validate whether the clusters are truly “industry clusters” vs “style/risk clusters” vs something more idiosyncratic?


## Main Question

Up to this point, you have constructed a simple two-factor PCA model and explored its loadings and time-series behavior. The number of factors was chosen arbitrarily, and no justification has been provided for why two factors should adequately describe the cross-section of returns (or why subsequently clustered into four groups).

Your task now is to continue trying to find interesting structure by exploring the data using unsupervised learning.

There is no single correct answer. A strong solution will:

- use diagnostics to justify key design choices (number of factors, number of clusters, etc.),
- explore at least **two** unsupervised approaches (PCA + one other, or PCA + clustering variants),
- discuss stability over time (do conclusions depend on the window?), and
- connect results to a business use-case (risk, diversification, monitoring, factor tilts, etc.).

---

**Question 6.** Starting from the baseline PCA workflow above:

1. Decide on a sensible number of PCA factors and justify your choice.
2. Use PCA outputs to explore cross-sectional structure. Then go beyond PCA using at least **one** additional unsupervised method (examples: clustering on loadings, clustering on correlations, hierarchical clustering, anomaly detection, etc.).
3. Provide evidence about whether the structure you find is stable (e.g., compare pre- and post-2020, or run PCA/clustering on subperiods).

Be explicit about:

- what your unsupervised method is optimizing,
- what “success” means in your context,
- and what you would want to validate using additional data sources.

---

**Question 7.** Finally (and most importantly), prepare a slide deck suitable for a 10 minute presentation to a portfolio manager or risk committee. Your presentation should include, at a minimum, 1. The business objective: why we are doing unsupervised learning here? 2. The main structure you found (factors, clusters, regimes) and how you diagnosed it. 3. How a manager could *use* these outputs operationally (risk monitoring, diversification checks, factor-aware allocation, stress testing, etc.). 4. What could go wrong (nonstationarity/regime changes, over-interpretation of PCA, sensitivity to sample window, and data leakage risks if used for prediction).


# Case 2. Household Segmentation for Pricing and Marketing


## Set-up

Imagine you work on an analytics team for a consumer packaged goods company that sells a premium dishwasher detergent.  The company has run a conjoint-style pricing study in which households were shown the product at different prices and asked whether they would purchase.

Management does not want a single "average" pricing recommendation.  Instead, they want to understand whether there are meaningfully different types of households in the market.  If such segments exist, the firm could use them to think about pricing, promotional targeting, channel strategy, and messaging.

Your job is to use unsupervised learning to search for useful structure in the household data.

### Data

Each row in the dataset represents a household/price combination. That is, each row corresponds to a household asked if they would purchase the product at a given price. Each household was asked the yes/no purchase at question at several prices, so each household appears in the raw data multiple times.

## Variables

- **`purchase`**  
  = 1 if the household is willing to purchase the detergent at that price
  = 0 otherwise  

- **`price`**  
  The price of the product.

- **`income_bracket`**  
  Household income bracket.

- **`age`**  
  Age of the household head.

- **`family_size`**  
  Number of people in the household.

- **`prev_interactions`**  
  Number of prior interactions with the brand

- **`state`**  
  U.S. state of residence.

- **`sex`**  
  Sex of household head.

- **`race`**  
  Race category of household head.

- **`household_id`**  
  Household identifier.



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import pairwise_distances
from sklearn.metrics import silhouette_score, silhouette_samples

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

np.random.seed(42)

## Loading and constructing the household-level dataset

The raw dataset contains repeated observations for households at different prices.  For clustering, it is more natural to work at the household level rather than at the household-price-row level. The code below creates a simple household-level table.  

In [ ]:
DROPBOX_CSV_URL = "https://www.dropbox.com/scl/fi/qzvgjnwdwipv4tkah6qvl/pricing_application_conjoint.csv?rlkey=l9sdcy2hw9daa3iv4491jp0je&dl=1"

df_raw = pd.read_csv(DROPBOX_CSV_URL)

for col in ["Unnamed: 0", "index", "Index"]:
    if col in df_raw.columns:
        df_raw = df_raw.drop(columns=col)

reservation_price = (
    df_raw.loc[df_raw["purchase"] == 1]
    .groupby("household_id")["price"]
    .max()
    .rename("reservation_price")
)

households = (
    df_raw.drop_duplicates("household_id")
    .merge(reservation_price, on="household_id", how="left")
)

households["reservation_price"] = households["reservation_price"].fillna(10)

households.head()


In [ ]:
exclude = ["purchase", "price"]
if "household_id" in households.columns:
    exclude.append("household_id")
if "hh_id" in households.columns:
    exclude.append("hh_id")

X = households.drop(columns=[c for c in exclude if c in households.columns]).copy()

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)
print("Feature matrix shape:", X.shape)


---

**Question 1.** Explain what the code above is doing.  Why is it reasonable to cluster households rather than household-price rows?  What is the interpretation of the constructed variable `reservation_price`, and what are its limitations as a business measure?

---


## Similarity / distance metrics

We need a notion of 'closeness' between observations. We'll start with numeric variables only to keep it simple, then add categorical variables.

### A warm-up: Distances between two households

We'll pick two random observations and compute distances under different metrics.

In [ ]:
# Pick two observations to compare
i, j = np.random.choice(len(X), size=2, replace=False)
row_i = X.iloc[i]
row_j = X.iloc[j]
row_i, row_j


In [ ]:
# We'll compute distances using numeric columns only (for now)
xi = row_i[num_cols].to_numpy().astype(float)
xj = row_j[num_cols].to_numpy().astype(float)

def l2(a, b):
    """Euclidean distance: straight-line distance between two points."""
    return float(np.sqrt(np.sum((a - b)**2)))

def l1(a, b):
    """Manhattan distance: total absolute difference across features."""
    return float(np.sum(np.abs(a - b)))

def linf(a, b):
    """Max (L-infinity) distance: largest difference on any single feature."""
    return float(np.max(np.abs(a - b)))

print("L2 (Euclidean):", l2(xi, xj))
print("L1 (Manhattan):", l1(xi, xj))
print("L∞ (Sup):", linf(xi, xj))


---

**Question 2.** Which variable(s) dominate these distances right now? Why?
What will happen to these distances if we standardize numeric variables first?

---

### A baseline representation

Many unsupervised methods are sensitive to scale and to the way categorical variables are encoded.  As a simple default, we will:

1. standardize numeric variables,
2. one-hot encode categorical variables, and
3. use that representation for both visualization and clustering.

This is a useful baseline, but it is not automatically the "right" notion of similarity for the business problem.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop",
)

X_rep = preprocessor.fit_transform(X)

print("Representation shape:", X_rep.shape)


---

**Question 3.** Why might standardization matter especially strongly in a clustering problem?  More broadly, does the representation above make an implicit judgment about what it means for two households to be "similar"?  Briefly explain.

Think about the variable `state`. After one-hot encoding, what are we implicitly saying about distances between states? Suggest an alternative way to capture the categorical variable `state`. (HINT: You can think about augmenting the data.) More generally, comment on the pros and cons of the practice of one-hot encoding categorical variables for use in clustering.

---


## Baseline clustering: K-means

We now build a simple segmentation using K-means.  As in the other cases, the point is to create a baseline result that you can critique and extend.

We fit K-means for several values of `K` and inspect two common diagnostics:

- **Inertia**: lower means tighter clusters.
- **Silhouette score**: higher means better separation.


In [ ]:
def fit_kmeans_grid(Xmat, k_grid, random_state=42, n_init="auto"):
    rows = []
    models = {}
    for k in k_grid:
        km = KMeans(n_clusters=k, random_state=random_state, n_init=n_init)
        labels = km.fit_predict(Xmat)
        rows.append({
            "K": k,
            "inertia": km.inertia_,
            "silhouette": silhouette_score(Xmat, labels)
        })
        models[k] = km
    return pd.DataFrame(rows), models

k_grid = list(range(2, 9))
km_table, km_models = fit_kmeans_grid(X_rep, k_grid)
km_table


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(km_table["K"], km_table["inertia"], marker="o")
ax[0].set_title("Elbow plot")
ax[0].set_xlabel("K")
ax[0].set_ylabel("Inertia")
ax[0].grid(True)

ax[1].plot(km_table["K"], km_table["silhouette"], marker="o")
ax[1].set_title("Silhouette score")
ax[1].set_xlabel("K")
ax[1].set_ylabel("Silhouette")
ax[1].grid(True)

plt.show()



---

**Question 4.** Choose a baseline value of `K`.  Justify your choice using the diagnostics above, but also explain why selecting `K` is ultimately a business judgment rather than a purely mechanical exercise.

---



## Inspecting one baseline segmentation

The next cell fixes a baseline number of clusters and produces simple summaries.  You should feel free to change `K_BASELINE`.

In [ ]:
K_BASELINE = 3

km = km_models[K_BASELINE]
labels = km.labels_

profile = households.copy()
profile["cluster"] = labels

print("Cluster sizes:")
print(profile["cluster"].value_counts().sort_index())

cluster_means = profile.groupby("cluster")[[c for c in num_cols if c in profile.columns]].mean()
cluster_means


---

**Question 5.** Give each baseline cluster a short business label.  For example, you might think in terms of price sensitivity, purchase propensity, or household characteristics.  Which cluster seems most commercially interesting, and why?


---


## Main Question Prompt

Up to this point, you have built a simple household-level segmentation using a particular representation and a baseline K-means workflow.  None of these design choices should be treated as final.

Your task now is to use unsupervised learning to produce a segmentation or low-dimensional structure that would actually be useful to a business stakeholder.

There is no single correct answer.  A strong solution will:

- think carefully about what the relevant object of segmentation is,
- justify the representation of households,
- use diagnostics to motivate major design choices,
- go beyond the baseline workflow,
- and connect the output to a concrete business use-case.

---

**Question 6.** Starting from the baseline analysis above, develop an improved unsupervised analysis of the market.  Your answer should do all of the following:

1. Decide what representation of households you want to use and explain why it is appropriate for the pricing / targeting problem.
2. Extend the analysis beyond the exact baseline shown above.  Examples include: an alternative clustering method, a different encoding of similarity, using PCA to explore the structure and/or aid in visualization, hierarchical clustering, or another sensible unsupervised approach.
3. Provide some evidence on whether your conclusions are stable.  For example, you might vary the feature construction, vary `K`, compare algorithms, or examine whether business conclusions are sensitive to modeling choices.
4. Translate the results into a business recommendation.  What would the firm actually do differently if it adopted your segmentation?

Be explicit about:

- what your method is optimizing,
- what "success" means in this context,
- and what additional data would help validate your conclusions.

---

**Question 7.** Finally (and most importantly), prepare a slide deck suitable for a 10 minute presentation to a pricing manager, brand manager, or marketing leadership team. You should imagine that the audience is chiefly interested in the structure you uncover and the business implications but also wants to understand how conclusions are being generated. Your presentation should include, at a minimum:

1. The business objective: why unsupervised learning is useful here.
2. The segmentation or structure you found and how you diagnosed it.
3. The commercial implication: pricing, promotions, targeting, product positioning, or experimentation.
4. What could go wrong: instability, overly mechanical segmentation, weak business actionability, or sensitivity to feature choices.
